## Import Required Libries

In [1]:
import numpy as np
import pandas as pd
import os
import joblib
import warnings
warnings.filterwarnings("ignore")

## Import DaaSet

In [2]:
data = pd.read_csv('../Data/Data.csv')

## Extract Information about DataSet

In [3]:
data.head()

,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP,image_url
0,BMW,1 Series M,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,"Factory Tuner,Luxury,High-Performance",Compact,Coupe,26,19,3916,46135,https://source.unsplash.com/600x400/?BMW+1+Ser...
1,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,28,19,3916,40650,https://source.unsplash.com/600x400/?BMW+1+Ser...
2,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,28,20,3916,36350,https://source.unsplash.com/600x400/?BMW+1+Ser...
3,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Coupe,28,18,3916,29450,https://source.unsplash.com/600x400/?BMW+1+Ser...
4,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,28,18,3916,34500,https://source.unsplash.com/600x400/?BMW+1+Ser...


In [4]:
data.shape

(11914, 17)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11914 entries, 0 to 11913
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Make               11914 non-null  object 
 1   Model              11914 non-null  object 
 2   Year               11914 non-null  int64  
 3   Engine Fuel Type   11911 non-null  object 
 4   Engine HP          11845 non-null  float64
 5   Engine Cylinders   11884 non-null  float64
 6   Transmission Type  11914 non-null  object 
 7   Driven_Wheels      11914 non-null  object 
 8   Number of Doors    11908 non-null  float64
 9   Market Category    8172 non-null   object 
 10  Vehicle Size       11914 non-null  object 
 11  Vehicle Style      11914 non-null  object 
 12  highway MPG        11914 non-null  int64  
 13  city mpg           11914 non-null  int64  
 14  Popularity         11914 non-null  int64  
 15  MSRP               11914 non-null  int64  
 16  image_url          119

In [6]:
data.describe()

,Year,Engine HP,Engine Cylinders,Number of Doors,highway MPG,city mpg,Popularity,MSRP
count,11914.000000,11845.00000,11884.000000,11908.000000,11914.000000,11914.000000,11914.000000,1.191400e+04
mean,2010.384338,249.38607,5.628829,3.436093,26.637485,19.733255,1554.911197,4.059474e+04
std,7.579740,109.19187,1.780559,0.881315,8.863001,8.987798,1441.855347,6.010910e+04
min,1990.000000,55.00000,0.000000,2.000000,12.000000,7.000000,2.000000,2.000000e+03
25%,2007.000000,170.00000,4.000000,2.000000,22.000000,16.000000,549.000000,2.100000e+04
50%,2015.000000,227.00000,6.000000,4.000000,26.000000,18.000000,1385.000000,2.999500e+04
75%,2016.000000,300.00000,6.000000,4.000000,30.000000,22.000000,2009.000000,4.223125e+04
max,2017.000000,1001.00000,16.000000,4.000000,354.000000,137.000000,5657.000000,2.065902e+06


## Remove the Useless Columns

In [7]:
data.drop(['Number of Doors', 'image_url'], axis = 1, inplace = True)

## Checking Null Value

In [8]:
data.isna().sum()

Make                    0
Model                   0
Year                    0
Engine Fuel Type        3
Engine HP              69
Engine Cylinders       30
Transmission Type       0
Driven_Wheels           0
Market Category      3742
Vehicle Size            0
Vehicle Style           0
highway MPG             0
city mpg                0
Popularity              0
MSRP                    0
dtype: int64

## Handling Null Values

In [9]:
for i in data.columns:
    if data[i].dtype == 'object':
        data[i] = data[i].fillna(data[i].mode()[0])
    else:
        data[i] = data[i].fillna(data[i].mean())

In [10]:
data.isna().sum()

Make                 0
Model                0
Year                 0
Engine Fuel Type     0
Engine HP            0
Engine Cylinders     0
Transmission Type    0
Driven_Wheels        0
Market Category      0
Vehicle Size         0
Vehicle Style        0
highway MPG          0
city mpg             0
Popularity           0
MSRP                 0
dtype: int64

## Merge the Columns

In [11]:
data['tag'] = ''
for i in data.columns:
    data['tag'] += data[i].astype(str) + ' ' 

## Seprating Independent Variable

In [12]:
X = data['tag']

## Encode the DataSet

### Tokenizer

In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer
token = Tokenizer(num_words = 1000, oov_token = "<OOV>")
token.fit_on_texts(X)
X = token.texts_to_sequences(X)

### Padding

In [14]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X = pad_sequences(X, maxlen = 100, padding = "post", truncating = "post")
print("Input shape:", X.shape)

Input shape: (11914, 100)


## Normalize

In [15]:
X = X / X.max()

## Create CNN Model

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, Dense, Dropout, Embedding, GlobalMaxPooling1D, GlobalAveragePooling1D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K

In [17]:
K.clear_session()
model = Sequential()
model.add(Input(shape=(X.shape[1],)))
model.add(Dense(512, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(Dense(256, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(Dense(128, activation='relu'))
model.add(BatchNormalization())
model.add(Dense(64, activation='relu', name='car_embedding'))
model.add(Dense(128, activation='relu'))
model.add(BatchNormalization())
model.add(Dense(256, activation='relu'))
model.add(BatchNormalization())
model.add(Dense(512, activation='relu'))
model.add(BatchNormalization())
model.add(Dense(100, activation='sigmoid'))

## Compile Model

In [18]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mae',
    metrics=['mae']
)

In [19]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 512)                 │          51,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 512)                 │           2,048 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ car_embedding (Dense)                │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 128)                 │           8,320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 256)                 │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 512)                 │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 512)                 │           2,048 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 100)                 │          51,300 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 455,588 (1.74 MB)

 Trainable params: 452,004 (1.72 MB)

 Non-trainable params: 3,584 (14.00 KB)

## Training the Model

In [20]:
history = model.fit(X, X, validation_split = 0.2, epochs=50, batch_size=32)

Epoch 1/50
298/298 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - loss: 0.1183 - mae: 0.1183 - val_loss: 0.0205 - val_mae: 0.0205
Epoch 2/50
298/298 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - loss: 0.0456 - mae: 0.0456 - val_loss: 0.0224 - val_mae: 0.0224
Epoch 3/50
298/298 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - loss: 0.0167 - mae: 0.0167 - val_loss: 0.0144 - val_mae: 0.0144
Epoch 4/50
298/298 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - loss: 0.0153 - mae: 0.0153 - val_loss: 0.0125 - val_mae: 0.0125
Epoch 5/50
298/298 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - loss: 0.0142 - mae: 0.0142 - val_loss: 0.0122 - val_mae: 0.0122
Epoch 6/50
298/298 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - loss: 0.0134 - mae: 0.0134 - val_loss: 0.0125 - val_mae: 0.0125
Epoch 7/50
298/298 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.0128 - mae: 0.0128 - val_loss: 0.0138 - val_mae: 0.0138
Epoch 8/50
298/298 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.0123 - mae: 0.0123 - val_loss: 0.0117 - val_mae: 0.0117
Epoch 9/50
298/298 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms

## Embedding the Model

In [21]:
embedding_model = Model(
    inputs=model.inputs,
    outputs=model.get_layer("car_embedding").output
)
car_embeddings = embedding_model.predict(X)
print("Car Embedding Shape:", car_embeddings)

373/373 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
Car Embedding Shape: [[0.         0.         0.         ... 1.3336847  2.1777368  0.        ]
 [0.         0.51547974 4.6399136  ... 0.         2.3807998  0.        ]
 [0.         0.         3.443357   ... 0.         2.7243686  0.        ]
 ...
 [0.         1.9581519  2.8291426  ... 0.         0.5976635  0.        ]
 [0.         1.980147   2.8210337  ... 0.         0.6208727  0.        ]
 [1.7150894  4.0539775  3.1350229  ... 0.00909379 0.         0.        ]]


## Creating the Similarity Matrix

In [22]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(car_embeddings)

## Saving the Similarity Matrix

In [23]:
import joblib
joblib.dump(similarity_matrix, 'Models/Recommandation_similarity_matrix.pkl')

['Models/Recommandation_similarity_matrix.pkl']

In [24]:
np.savez_compressed("Models/Recommandation_similarity_matrix.npz", similarity_matrix)

## Saving the Car Embeddings

In [25]:
joblib.dump(car_embeddings, 'Models/car_embeddings.pkl')

['Models/car_embeddings.pkl']